# Ad Creative Quality Scorer — PyTorch
ResNet-50 + BiLSTM + ONNX export


In [1]:
!git clone https://github.com/saitejasrivilli/ad-creative-scorer.git
import os, sys
os.chdir("/content/ad-creative-scorer")
sys.path.insert(0, os.getcwd())
os.makedirs("models", exist_ok=True)
print(os.listdir("."))

Cloning into 'ad-creative-scorer'...
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 18 (delta 1), reused 18 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (18/18), 22.70 KiB | 5.68 MiB/s, done.
Resolving deltas: 100% (1/1), done.
['.DS_Store', 'train_colab.ipynb', 'scorer.py', 'scorer.cpp', '.git', 'README.md', 'notebooks', 'api.py', 'onnx_export.py', 'synthetic_data.py', 'model', 'requirements.txt', 'models']


In [2]:
!pip install -q torch torchvision onnx onnxruntime numpy pandas scikit-learn
import torch
print("PyTorch:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 63.1 MB/s eta 0:00:00
PyTorch: 2.10.0+cu128
Device: cuda


In [4]:
import os
print(os.listdir('/content/ad-creative-scorer'))

['.DS_Store', 'train_colab.ipynb', 'scorer.py', 'scorer.cpp', '.git', 'README.md', 'notebooks', 'api.py', 'onnx_export.py', 'synthetic_data.py', 'model', 'requirements.txt', 'models']


In [5]:
import os, sys
# Try to find the right directory
for path in [
    '/content/ad-creative-scorer',
    '/content/ad-creative-scorer/ad-creative-scorer',
]:
    if os.path.exists(path + '/data'):
        os.chdir(path)
        break

sys.path.insert(0, os.getcwd())
print('cwd:', os.getcwd())
print('contents:', os.listdir('.'))

cwd: /content/ad-creative-scorer
contents: ['.DS_Store', 'train_colab.ipynb', 'scorer.py', 'scorer.cpp', '.git', 'README.md', 'notebooks', 'api.py', 'onnx_export.py', 'synthetic_data.py', 'model', 'requirements.txt', 'models']


In [7]:
!git -C /content/ad-creative-scorer pull --force
import os, sys
os.chdir('/content/ad-creative-scorer')
sys.path.insert(0, os.getcwd())
print(os.listdir('.'))

remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 8 (delta 1), reused 8 (delta 1), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 1.12 KiB | 573.00 KiB/s, done.
From https://github.com/saitejasrivilli/ad-creative-scorer
   a7260fa..6093331  main       -> origin/main
Updating a7260fa..6093331
Fast-forward
 .DS_Store                                   | Bin 6148 -> 6148 bytes
 data/__init__.py                            |   0
 synthetic_data.py => data/synthetic_data.py |   0
 export/__init__.py                          |   0
 onnx_export.py => export/onnx_export.py     |   0
 postprocess/__init__.py                     |   0
 scorer.cpp => postprocess/scorer.cpp        |   0
 scorer.py => postprocess/scorer.py          |   0
 serving/__init__.py                         |   0
 api.py => serving/api.py                    |   0
 10 files changed, 0 insertions(+), 0 deletions(-)
 create mod

In [9]:
exec(open("data/synthetic_data.py").read())
import numpy as np
df, img_feats, txt_tokens = generate_dataset(n_samples=20000)
(tr_df,tr_img,tr_txt),(va_df,va_img,va_txt),(te_df,te_img,te_txt) = train_val_test_split(df,img_feats,txt_tokens)
print(f"Train: {len(tr_df)} | Val: {len(va_df)} | Test: {len(te_df)}")

Generated 10,000 ad creatives
Category distribution:
category
brand              2066
product            2026
lifestyle          2024
video_thumbnail    1952
text_heavy         1932
Mean quality score: 0.640
Mean CTR: 0.0768
Train: 7,000 | Val: 1,500 | Test: 1,500

✓ Dataset generation complete
Generated 20,000 ad creatives
Category distribution:
category
brand              4077
text_heavy         4057
product            4029
lifestyle          3971
video_thumbnail    3866
Mean quality score: 0.640
Mean CTR: 0.0769
Train: 14,000 | Val: 3,000 | Test: 3,000
Train: 14000 | Val: 3000 | Test: 3000


In [10]:
exec(open("model/multitask.py").read())
model = CreativeScorer().to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Parameters: 1,445,895
Quality: torch.Size([8, 1]), Category: torch.Size([8, 5])
✓ Smoke test passed
Parameters: 1,445,895


In [11]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder().fit(["product","lifestyle","text_heavy","video_thumbnail","brand"])

def make_loader(df, img, txt, shuffle=True, bs=256):
    q = torch.tensor(df["quality_score"].values, dtype=torch.float32)
    c = torch.tensor(le.transform(df["category"].values), dtype=torch.long)
    i = torch.tensor(img, dtype=torch.float32)
    t = torch.tensor(txt, dtype=torch.long)
    return DataLoader(TensorDataset(i,t,q,c), batch_size=bs, shuffle=shuffle)

train_dl = make_loader(tr_df, tr_img, tr_txt)
val_dl   = make_loader(va_df, va_img, va_txt, shuffle=False)
test_dl  = make_loader(te_df, te_img, te_txt, shuffle=False)
print(f"Batches: train={len(train_dl)} val={len(val_dl)} test={len(test_dl)}")

Batches: train=55 val=12 test=12


In [12]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5)
best_corr = 0; history = {"train_loss":[], "val_corr":[]}

for epoch in range(20):
    model.train(); total_loss = 0
    for img_b,txt_b,q_b,c_b in train_dl:
        img_b,txt_b,q_b,c_b = img_b.to(device),txt_b.to(device),q_b.to(device),c_b.to(device)
        optimizer.zero_grad()
        q_pred,c_pred = model(img_b,txt_b)
        loss,_ = model.compute_loss(q_pred,c_pred,q_b,c_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()

    model.eval(); preds,trues = [],[]
    with torch.no_grad():
        for img_b,txt_b,q_b,c_b in val_dl:
            q_p,_ = model(img_b.to(device),txt_b.to(device))
            preds.extend(q_p.cpu().squeeze().tolist())
            trues.extend(q_b.tolist())
    import numpy as np
    corr = float(np.corrcoef(preds,trues)[0,1])
    history["train_loss"].append(total_loss/len(train_dl))
    history["val_corr"].append(corr)
    print(f"Epoch {epoch+1:2d} | loss={total_loss/len(train_dl):.4f} | val_corr={corr:.4f}")
    if corr > best_corr:
        best_corr = corr
        torch.save(model.state_dict(), "models/creative_scorer_best.pt")
        print(f"  Saved best (corr={corr:.4f})")

Epoch  1 | loss=0.4875 | val_corr=0.8725
  Saved best (corr=0.8725)
Epoch  2 | loss=0.2368 | val_corr=0.8790
  Saved best (corr=0.8790)
Epoch  3 | loss=0.1213 | val_corr=0.8838
  Saved best (corr=0.8838)
Epoch  4 | loss=0.0594 | val_corr=0.8944
  Saved best (corr=0.8944)
Epoch  5 | loss=0.0354 | val_corr=0.8937
Epoch  6 | loss=0.0548 | val_corr=0.8969
  Saved best (corr=0.8969)
Epoch  7 | loss=0.0896 | val_corr=0.8996
  Saved best (corr=0.8996)
Epoch  8 | loss=0.0508 | val_corr=0.9043
  Saved best (corr=0.9043)
Epoch  9 | loss=0.0274 | val_corr=0.9092
  Saved best (corr=0.9092)
Epoch 10 | loss=0.0176 | val_corr=0.9100
  Saved best (corr=0.9100)
Epoch 11 | loss=0.0200 | val_corr=0.9139
  Saved best (corr=0.9139)
Epoch 12 | loss=0.0450 | val_corr=0.9121
Epoch 13 | loss=0.0448 | val_corr=0.9121
Epoch 14 | loss=0.0217 | val_corr=0.9160
  Saved best (corr=0.9160)
Epoch 15 | loss=0.0165 | val_corr=0.9167
  Saved best (corr=0.9167)
Epoch 16 | loss=0.0175 | val_corr=0.9175
  Saved best (corr=0

In [13]:
model.load_state_dict(torch.load("models/creative_scorer_best.pt", map_location=device))
model.eval()
preds,trues,cat_preds,cat_trues = [],[],[],[]
with torch.no_grad():
    for img_b,txt_b,q_b,c_b in test_dl:
        q_p,c_p = model(img_b.to(device),txt_b.to(device))
        preds.extend(q_p.cpu().squeeze().tolist())
        trues.extend(q_b.tolist())
        cat_preds.extend(c_p.cpu().argmax(1).tolist())
        cat_trues.extend(c_b.tolist())
corr = float(np.corrcoef(preds,trues)[0,1])
cat_acc = float(np.mean(np.array(cat_preds)==np.array(cat_trues)))
print("=== TEST RESULTS ===")
print(f"  Quality correlation (r): {corr:.4f}")
print(f"  Category accuracy:       {cat_acc:.4f}")
print(f"  Best val correlation:    {best_corr:.4f}")

=== TEST RESULTS ===
  Quality correlation (r): 0.9160
  Category accuracy:       0.7607
  Best val correlation:    0.9207


In [17]:
import time
import torch

import warnings, logging
logging.getLogger("onnxscript").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

torch.onnx.export(
    model_cpu,
    (dummy_img, dummy_txt),
    "models/creative_scorer.onnx",
    input_names=["image_features", "text_tokens"],
    output_names=["quality_score", "category_logits"],
    dynamic_axes={"image_features": {0: "batch"}, "text_tokens": {0: "batch"}},
)
print("ONNX exported")
warnings.resetwarnings()
# Benchmark PyTorch
lats = []
for _ in range(200):
    t0 = time.perf_counter()
    with torch.no_grad(): _ = model_cpu(dummy_img, dummy_txt)
    lats.append((time.perf_counter()-t0)*1000)
lats.sort()
print(f"PyTorch p50={lats[100]:.3f}ms p95={lats[190]:.3f}ms p99={lats[198]:.3f}ms")

# Benchmark ONNX
import onnxruntime as ort
sess = ort.InferenceSession("models/creative_scorer.onnx", providers=["CPUExecutionProvider"])
img_np = dummy_img.numpy(); txt_np = dummy_txt.numpy()
onnx_lats = []
for _ in range(200):
    t0 = time.perf_counter()
    _ = sess.run(None, {"image_features": img_np, "text_tokens": txt_np})
    onnx_lats.append((time.perf_counter()-t0)*1000)
onnx_lats.sort()
print(f"ONNX    p50={onnx_lats[100]:.3f}ms p95={onnx_lats[190]:.3f}ms p99={onnx_lats[198]:.3f}ms")
speedup = lats[198]/onnx_lats[198]
print(f"Speedup p99: {speedup:.2f}x | Reduction: {(1-onnx_lats[198]/lats[198])*100:.1f}%")

[torch.onnx] Obtain model graph for `CreativeScorer([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `CreativeScorer([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 3 of general pattern rewrite rules.
ONNX exported


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


PyTorch p50=1.713ms p95=2.603ms p99=2.944ms
ONNX    p50=0.574ms p95=0.695ms p99=0.826ms
Speedup p99: 3.56x | Reduction: 71.9%


In [18]:
import shutil
shutil.make_archive("creative_scorer_artifacts","zip","models/")
from google.colab import files
files.download("creative_scorer_artifacts.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>